# 19 — Time Series Resampling, Stock vs Flow Aggregations, & Per-Capita Analytics
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to time series frequency conversion, datetime index management, and handling domain metrics safely in Python.*

---

## 📌 Executive Summary & Interview Expectations
In analytics and data engineering interviews, working with time-series aggregates is a frequent screening topic. Interviewers use datasets like US Crime Rates to test two critical capabilities:
1. **Datetime Index Mechanics & Resampling**: Converting date representations, configuring a `DatetimeIndex`, and downsampling with modern frequency strings (e.g. `'10YS'` replacing deprecated `'10AS'`).
2. **The "Stock vs Flow" Aggregation Trap**: Recognizing that flow metrics (crime counts, transactions, revenue) should be **summed**, while stock metrics (population, bank balances, inventory) represent a snapshot state and must **never be summed across time periods**.
3. **Data Integrity & Metric Normalization**: Identifying partial time intervals (e.g. the 2010s decade only containing 5 years) and calculating **per-capita metrics ($per\ 100,000$)** rather than comparing raw gross counts across changing populations.

## 1. Environment Setup & Data Ingestion

In [1]:
import numpy as np
import pandas as pd

# Load US Crime Rates dataset
crime = pd.read_csv("US_Crime_Rates_1960_2014.csv")
print(f"Dataset Loaded: {crime.shape[0]} years of data (Shape: {crime.shape})")
crime.head(3)

Dataset Loaded: 55 years of data (Shape: (55, 12))


,Year,Population,Total,Violent,Property,Murder,Forcible_Rape,Robbery,Aggravated_assault,Burglary,Larceny_Theft,Vehicle_Theft
0,1960,179323175,3384200,288460,3095700,9110,17190,107840,154320,912100,1855400,328200
1,1961,182992000,3488000,289390,3198600,8740,17220,106670,156760,949600,1913000,336000
2,1962,185771000,3752200,301510,3450700,8530,17550,110860,164570,994300,2089600,366800


## 2. Datetime Parsing & DatetimeIndex Setup

### 💡 Interview Tip: Explicit Formatting in `pd.to_datetime`
- Always specify `format='%Y'` when parsing year integers or strings.
- Specifying the format avoids expensive heuristic parsing and protects against ambiguous date interpretation.

In [2]:
# Check initial data types
print("Initial Data Types:")
print(crime.dtypes.head(4))

# Convert Year integer to datetime64[ns]
crime["Year"] = pd.to_datetime(crime["Year"], format="%Y")

# Set Year as the DatetimeIndex immutably
crime = crime.set_index("Year")
print("\nUpdated Index Type:", type(crime.index))
crime.head(3)

Initial Data Types:
Year          int64
Population    int64
Total         int64
Violent       int64
dtype: object

Updated Index Type: <class 'pandas.DatetimeIndex'>


,Population,Total,Violent,Property,Murder,Forcible_Rape,Robbery,Aggravated_assault,Burglary,Larceny_Theft,Vehicle_Theft
Year,,,,,,,,,,,
1960-01-01,179323175,3384200,288460,3095700,9110,17190,107840,154320,912100,1855400,328200
1961-01-01,182992000,3488000,289390,3198600,8740,17220,106670,156760,949600,1913000,336000
1962-01-01,185771000,3752200,301510,3450700,8530,17550,110860,164570,994300,2089600,366800


## 3. Schema Pruning: Dropping Redundant Columns

### 💡 Interview Note: `del` vs `.drop()`
- `del df['Total']`: Modifies the DataFrame in-place via Python statement; does not return anything and cannot be chained.
- `df.drop(columns=['Total'])`: Idiomatic, functional, supports method chaining, and can accept multiple columns with error handling (`errors='ignore'`).

In [3]:
# Drop the pre-aggregated 'Total' column to prevent double-counting
crime = crime.drop(columns=["Total"])
print("Columns after dropping 'Total':")
print(crime.columns.tolist())

Columns after dropping 'Total':
['Population', 'Violent', 'Property', 'Murder', 'Forcible_Rape', 'Robbery', 'Aggravated_assault', 'Burglary', 'Larceny_Theft', 'Vehicle_Theft']


## 4. Decade Resampling & The Stock vs Flow Trap

### 🚨 Top Interview Gotcha: The 2-Billion Population Blunder!
- If an engineer executes: `crimes = crime.resample('10YS').sum()`, look at the `'Population'` column!
- Adding the US population of ~200M every year for 10 years produces **~2 Billion people per decade**!
- **Domain Rule**:
  - **Flow Variable**: Crimes committed per year $\rightarrow$ Accumulate via **`.sum()`**.
  - **Stock Variable**: Population at a point in time $\rightarrow$ Aggregate via **`.max()`**, **`.last()`**, or **`.mean()`**.

In [4]:
# Step 1: Resample flow variables (crime categories) with .sum()
# Using modern frequency '10YS' (10-Year Start)
crime_cols = [c for c in crime.columns if c != "Population"]
crimes_decades = crime[crime_cols].resample("10YS").sum()

# Step 2: Resample stock variable (Population) with .max() or .last()
population_decades = crime["Population"].resample("10YS").max()

# Step 3: Combine correctly
crimes_decades["Population"] = population_decades

print("Decade-Aggregated Data (1960 - 2014):")
display(crimes_decades)

Decade-Aggregated Data (1960 - 2014):


,Violent,Property,Murder,Forcible_Rape,Robbery,Aggravated_assault,Burglary,Larceny_Theft,Vehicle_Theft,Population
Year,,,,,,,,,,
1960-01-01,4134930,45160900,106180,236720,1633510,2158520,13321100,26547700,5292100,201385000
1970-01-01,9607930,91383800,192230,554570,4159020,4702120,28486000,53157800,9739900,220099000
1980-01-01,14074328,117048900,206439,865639,5383109,7619130,33073494,72040253,11935411,248239000
1990-01-01,17527048,119053499,211664,998827,5748930,10568963,26750015,77679366,14624418,272690813
2000-01-01,13968056,100944369,163068,922499,4230366,8652124,21565176,67970291,11412834,307006550
2010-01-01,6072017,44095950,72867,421059,1749809,3764142,10125170,30401698,3569080,318857056


## 5. Identifying the "Most Dangerous Decade": Raw vs Per-Capita

### ⚠️ Top Interview Question: Why Raw Totals Lie
1. **Changing Population**: US population grew from 179M (1960) to 318M (2014). More citizens naturally yield higher absolute crime counts even if society is safer!
2. **Incomplete Interval**: The `2010-01-01` bin only contains **5 years of data (2010–2014)**, whereas other bins contain 10 years.
3. **Metric Standardization**: To answer *"Which decade was most dangerous to live in?"*, you MUST compute **Crimes per 100,000 citizens**!

In [5]:
# 1. Raw Peak Decade (Unadjusted)
raw_peak_decade = crimes_decades["Violent"].idxmax()
print(f"Peak Decade by Raw Violent Crime Count: {raw_peak_decade.year}s ({crimes_decades.loc[raw_peak_decade, 'Violent']:,} crimes)")

# 2. Per-Capita Rate Calculation (Crimes per 100,000 citizens per year)
# Note: Normalize by number of years in the bin (10 years, except 2010-2014 which is 5 years)
years_per_decade = crime.resample("10YS").size()

# Annualized violent crime rate per 100,000
crimes_decades["Violent_Annual_Rate_per_100k"] = (
    (crimes_decades["Violent"] / years_per_decade) / crimes_decades["Population"]
) * 100_000

crimes_decades["Murder_Annual_Rate_per_100k"] = (
    (crimes_decades["Murder"] / years_per_decade) / crimes_decades["Population"]
) * 100_000

display(crimes_decades[["Population", "Violent", "Violent_Annual_Rate_per_100k", "Murder_Annual_Rate_per_100k"]])

per_capita_peak = crimes_decades["Violent_Annual_Rate_per_100k"].idxmax()
print(f"\nTrue Most Dangerous Decade Per-Capita: {per_capita_peak.year}s with {crimes_decades.loc[per_capita_peak, 'Violent_Annual_Rate_per_100k']:.1f} violent crimes per 100k people annually!")

Peak Decade by Raw Violent Crime Count: 1990s (17,527,048 crimes)


,Population,Violent,Violent_Annual_Rate_per_100k,Murder_Annual_Rate_per_100k
Year,,,,
1960-01-01,201385000,4134930,205.324627,5.272488
1970-01-01,220099000,9607930,436.527653,8.733797
1980-01-01,248239000,14074328,566.966834,8.316139
1990-01-01,272690813,17527048,642.744352,7.762051
2000-01-01,307006550,13968056,454.975830,5.311548
2010-01-01,318857056,6072017,380.861385,4.570512



True Most Dangerous Decade Per-Capita: 1990s with 642.7 violent crimes per 100k people annually!


## 6. Time Series Resampling Frequency Reference

| Frequency Code | Pandas 2.2+ Standard | Legacy Code (Deprecated) | Description |
| :--- | :--- | :--- | :--- |
| **Year Start** | `'YS'` | `'AS'` | Starts on Jan 1st of each year |
| **Year End** | `'YE'` | `'Y'` / `'A'` | Ends on Dec 31st of each year |
| **Quarter Start** | `'QS'` | `'QS'` | Starts on first day of quarter |
| **Month End** | `'ME'` | `'M'` | Ends on last day of month |
| **Business Month End**| `'BME'` | `'BM'` | Last business day of month |
| **Day** | `'D'` | `'D'` | Calendar day |
| **Hour** | `'h'` | `'H'` | Hour interval |

---
## 🎯 7. Technical Interview Corner: Tricky Questions & Drills

### Q1: What is the architectural difference between `.resample()` and `.groupby()`?
**Answer**:
- **`.groupby()`**: Groups purely by distinct discrete keys present in the data. If dates in 1975 are missing from the dataset, `.groupby(df.index.year)` will simply skip 1975 without creating an entry.
- **`.resample()`**: A frequency-based bucket generator that generates a **continuous regular time grid** across the entire span (`min_date` to `max_date`). Even if a bucket has 0 observations, `resample` produces the interval row (filled with `NaN` or 0 depending on the aggregation).

---

### Q2: How can you resample different columns with different aggregation functions in a single call?
**Answer**:
Pass an aggregation dictionary to `.resample().agg()`:
```python
df.resample('10YS').agg({
    'Violent': 'sum',
    'Property': 'sum',
    'Population': 'last'
})
```
This avoids creating intermediate dataframes or manual column stitching!

In [6]:
# Interview Demonstration: One-Shot Resampling with Custom Aggregation Map
clean_decade_summary = crime.resample("10YS").agg({
    "Population": "last",
    "Violent": "sum",
    "Property": "sum",
    "Murder": "sum",
    "Robbery": "sum"
})

print("One-Shot Resampling Result:")
display(clean_decade_summary)

One-Shot Resampling Result:


,Population,Violent,Property,Murder,Robbery
Year,,,,,
1960-01-01,201385000,4134930,45160900,106180,1633510
1970-01-01,220099000,9607930,91383800,192230,4159020
1980-01-01,248239000,14074328,117048900,206439,5383109
1990-01-01,272690813,17527048,119053499,211664,5748930
2000-01-01,307006550,13968056,100944369,163068,4230366
2010-01-01,318857056,6072017,44095950,72867,1749809
